# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions. Write your own code in the empty cells provided.
- Section **8** is pre-written for evaluator use and **must not be modified**. Organizers run it
  against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** (`Restart Kernel & Run All`) in
  **under 15 minutes**. Organizers will rerun it on their own machine, so saved outputs are not needed.
- **Scoring metric: F1-score** on the hidden test set (higher is better).

## Rules

- **Folder layout** (inside your team repository, `AI Task` folder):
  ```
  AI Task/
  ├── data/
  │   ├── task1_train.csv            <- your training data (keep it here)
  │   └── task1_evaluation_split.csv  <- added by the organizers when they rerun
  └── TeamName_Task1_CustomerChurn.ipynb
  ```
- Use **only the provided training file**. External datasets, hardcoded labels and lookup tricks
  lead to disqualification.
- Use `RANDOM_STATE` for every split and every model so results can be reproduced.
- Use only `pandas`, `numpy` and `scikit-learn` (`xgboost` / `lightgbm` are also fine).
- If Section 8 fails when the organizers rerun your notebook, you receive **zero Model Performance marks** for this task.

## Three golden rules

1. **Split first.** Split the raw data into train / validation / test (Section 2) before fitting anything.
2. **Encoders, scalers and the model go into one sklearn `Pipeline`** that is fitted on the **train split only**.
3. **`predict_*()` only calls the already-fitted pipeline.** Never refit anything on new data.

## 0. Setup

Standard imports, file paths and the metric function are set up below.
Add any additional imports your approach needs in the same cell.
`evaluate_predictions()` is defined here so it is available everywhere below. Do not modify it.

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# File paths (do not change; see the folder layout above)
TRAIN_PATH = "./data/task1_train.csv"
TARGET_COLUMN = "Churn"
EVALUATION_SPLIT_PATH = "./data/task1_evaluation_split.csv"  # organizers only: added at grading time

# Use this seed for every split and every model
RANDOM_STATE = 42

# Do not modify this function
def evaluate_predictions(y_true, y_pred, y_proba):
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Sections 6-7) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [14]:
from google.colab import files

uploaded = files.upload()

Saving dataset-1.csv to dataset-1 (1).csv


## 1. Load the Data

The training data is loaded from `TRAIN_PATH` for you.

## 2. Train / Validation / Test Split

Split the **raw** data right after loading it (before exploring, cleaning, scaling or any fitting) into three sets, for example 60% / 20% / 20%: Stratify on the target so every split has a similar churn rate.

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 7.

Use `random_state=RANDOM_STATE`. Then call it once, e.g. `train_split, validation_split, test_split = split_data(df)`. From here on, work with `train_split`; the other two are only used in Sections 6 and 7.

In [28]:
def split_data(df):
    """Split the raw dataframe into (train_split, validation_split, test_split)."""
    x=df.drop(columns=["Churn"])
    y=df["Churn"]
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
    x_train,x_val,y_train,y_val=train_test_split(x_train,y_train,test_size=0.25,random_state=42,stratify=y_train)

    raise NotImplementedError("Return train_split, validation_split, test_split")




## 3. Explore and Clean the Dataset

First explore the **train split** (looking only; do not fit anything yet), then write `data_cleaning()`.

**Explore:** missing values, categorical variables, summary statistics, class balance of the target. Check duplicate rows, inconsistent categories and outliers.

**`data_cleaning(df)` must be stateless and row-preserving:**

- Allowed: type conversion, fixing category spellings, replacing placeholder text with `NaN`.
- **Not allowed:** dropping rows, using the target column, or learning anything from the data (no `fit`).
- Missing values are filled later by a `SimpleImputer` inside the Pipeline (Section 5).
- `TotalCharges` is stored as text and can contain blank values. Convert it to numbers.
- The `customerID` column is **not** in the evaluation file, so do not use it as a feature.

**Dropping rows** (duplicates, outliers, rows with too many missing values) is done in a separate function, `remove_bad_training_rows()`, below. It is applied to the **train split only**, so the validation, test and hidden data keep all their rows (missing values there are filled by the imputer instead).
Document every row/column you drop and why in a markdown cell.

In [ ]:
# Exploratory data analysis: distributions, missing values, target balance, outliers, etc.

In [23]:
def data_cleaning(df):
    """Stateless, row-preserving cleaning. Must work with or without the target column."""
    num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])
    cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])
    # Implement your cleaning steps here
    return df

In [25]:
def remove_bad_training_rows(train_df):
    """Drop duplicates / outliers / unusable rows. Called on the TRAIN split only."""
    train_df = train_df.drop_duplicates()


    # Implement your row-removal steps here (you may use the target column)
    return train_df

## 4. Feature Engineering

Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, grouping `tenure` into buckets, or counting how many services a customer uses.

Like `data_cleaning()`, this function must not drop rows or use the target column.

In [ ]:
def feature_engineering(num_cols,cat_cols):
    """Stateless, row-preserving feature creation. Must work with or without the target column."""
    # Implement your feature engineering steps here

    num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

    cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())])


   preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

    return df

## 5. Transform, Normalize, and Scale Features

Build the (unfitted) preprocessing step that is used inside your Pipeline:

- Encode categorical variables (e.g. `OneHotEncoder(handle_unknown="ignore")`, so unseen categories in the hidden data do not crash).
- Scale numeric features if the model needs it (e.g. `StandardScaler`).
- Use `SimpleImputer` for missing values.
- Combine everything in a `ColumnTransformer` and **return it without fitting**. It is fitted in Section 6 on the train split only.

In [ ]:
def build_preprocessor():
    """Return an UNFITTED ColumnTransformer (imputing, encoding, scaling)."""
    # Implement your preprocessing steps here
    raise NotImplementedError("Build and return a ColumnTransformer")

In [ ]:
# Do not modify this function
def preprocess_data(df):
    n_rows = len(df)
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    assert len(df) == n_rows, "data_cleaning/feature_engineering must not drop rows"
    return df

## 6. Model Training and Hyperparameter Tuning

1. Prepare the data: `preprocess_data()` on each split, then separate features from the target. The target `Churn` is stored as `True`/`False`. Convert it to 0/1 for training.
2. Apply `remove_bad_training_rows()` to the **train split only** (never to validation, test or hidden data).
3. Build `Pipeline([("preprocessor", build_preprocessor()), ("model", ...)])` and fit it on the **train split only**.
4. Try more than one algorithm (Logistic Regression as a baseline, then a tree-based model) and compare them.
5. Tune hyperparameters using the **validation set** (e.g. `GridSearchCV`, `RandomizedSearchCV` or a manual sweep).
6. Use `evaluate_predictions()` (defined in Section 0) to report **train** and **validation** metrics for each model.
7. Store your best fitted pipeline in a variable called **`model`**.

### Required output of this section
- **`predict_churn(df)`** — takes a raw dataframe (no target column), calls `preprocess_data(df)` internally, and returns `(predictions, probabilities)` from your best fitted pipeline:
  `predictions` are **0/1 labels** and `probabilities` are the **probability of class 1 (churn)**.

This function is what Section 8 calls on the hidden test set. It must work on data that has not been touched yet, and it must not refit anything.

In [ ]:
# Train, compare and tune your models here.
# At the end, `model` must be your best fitted Pipeline.
model = None

In [ ]:
def predict_churn(df):
    """Return (0/1 predictions, probability of class 1) for a raw dataframe."""
    processed = preprocess_data(df)
    predictions = model.predict(processed)
    probabilities = model.predict_proba(processed)[:, 1]
    return predictions, probabilities

## 7. Evaluate on the Held-Out Test Set

Run `predict_churn()` on your held-out **test split** (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

Then write your **Approach summary** in the markdown cell below.

In [30]:
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

NameError: name 'y_test' is not defined

### Approach summary

*Replace this text with 5–8 lines in your own words: how you cleaned the data, which features you created and why, which models you compared, how you tuned them, and why you chose your final model.*

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.

In [ ]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
# Accept Yes/No, True/False or 1/0 labels
y_hidden = (
    evaluation_df[TARGET_COLUMN].astype(str).str.strip().str.lower()
    .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
)
assert y_hidden.notna().all(), "Unrecognised label values in the evaluation file"
y_hidden = y_hidden.astype(int)

y_pred, y_proba = predict_churn(X_hidden)
y_pred = np.asarray(y_pred).astype(int)
y_proba = np.asarray(y_proba)
assert len(y_pred) == len(y_hidden) == len(y_proba), (
    "predict_churn must return one prediction and one probability per row"
)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])